# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanhGiauTen/flyrankAI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

A transparent low-CTR opportunity rule for the content-review queue. The current-window decline label is used only for retrospective evaluation, never to build the score.

## 1. Check two signals, then state the rule

**Signal 1 — visibility/volume: MIXED.** Decline-proxy rates rise from very low volume into the middle bands, then fall again at 30k+ impressions. Volume identifies impact, but it is not a monotonic risk signal.

**Signal 2 — CTR gap within position band: CONFIRMED.** Among pages with at least 500 impressions and valid positions, decline-proxy rates rise across quartiles of the gap between position-band median CTR and observed CTR. This supports a simple opportunity rule while remaining only observational.

**Rule in plain words:** rank visible pages in positions 1–50 by the size of their CTR shortfall versus comparable position-band pages, weighted by log impressions. Every prioritized row receives the single reason code `low_ctr_visible_page` and the action `review title, snippet, and search-intent fit`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

DATA_URL = 'https://raw.githubusercontent.com/KhanhGiauTen/flyrankAI/main/data/raw/content_refresh_anonymized.csv'
local_candidates = [Path('data/raw/content_refresh_anonymized.csv'), Path('../../data/raw/content_refresh_anonymized.csv')]
data_source = next((path for path in local_candidates if path.exists()), DATA_URL)
df = pd.read_csv(data_source)
df['is_declining_proxy'] = df['trend_direction'].eq('down').astype('int8')

df['impression_band'] = pd.cut(
    df['impressions_90d'], [-1, 99, 499, 2999, 29999, np.inf],
    labels=['<100', '100-499', '500-2,999', '3k-29,999', '30k+']
)
volume_check = df.groupby('impression_band', observed=False).agg(
    n=('content_id', 'size'),
    decline_proxy_rate=('is_declining_proxy', 'mean'),
    median_sessions=('sessions_90d', 'median'),
)
print('SIGNAL 1 — VOLUME: MIXED')
display(volume_check.style.format({'decline_proxy_rate': '{:.1%}', 'median_sessions': '{:.0f}'}))

eligible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 50)].copy()
eligible['position_band'] = pd.cut(
    eligible['avg_position'], [0, 3, 10, 20, 50],
    labels=['top_3', 'page_1', 'striking', 'page_3_5'], include_lowest=True
)
position_medians = eligible.groupby('position_band', observed=True)['ctr'].median()
eligible['expected_ctr'] = eligible['position_band'].map(position_medians).astype(float)
eligible['ctr_gap'] = eligible['expected_ctr'] - eligible['ctr']
eligible['ctr_gap_quartile'] = pd.qcut(eligible['ctr_gap'], 4, duplicates='drop')
ctr_gap_check = eligible.groupby('ctr_gap_quartile', observed=True).agg(
    n=('content_id', 'size'),
    median_ctr_gap=('ctr_gap', 'median'),
    decline_proxy_rate=('is_declining_proxy', 'mean'),
)
print('SIGNAL 2 — POSITION-ADJUSTED CTR GAP: CONFIRMED')
display(ctr_gap_check.style.format({'median_ctr_gap': '{:.2f}', 'decline_proxy_rate': '{:.1%}'}))


SIGNAL 1 — VOLUME: MIXED


,n,decline_proxy_rate,median_sessions
impression_band,,,
<100,7994,38.9%,2
100-499,5280,60.4%,4
"500-2,999",8443,62.1%,9
"3k-29,999",7205,58.6%,41
30k+,1078,46.2%,184


SIGNAL 2 — POSITION-ADJUSTED CTR GAP: CONFIRMED


,n,median_ctr_gap,decline_proxy_rate
ctr_gap_quartile,,,
"(-5.261, -0.16]",4100,-0.36,50.8%
"(-0.16, 0.0]",4276,-0.06,59.4%
"(0.0, 0.09]",4437,0.06,62.1%
"(0.09, 0.24]",3524,0.17,70.0%


## 2. Build the ranked queue (writes the CSV)

The score is intentionally simple and unfitted: `log1p(impressions_90d) × max(position-band median CTR − observed CTR, 0)`. Log impressions rewards impact without allowing a single huge page to dominate linearly. The queue uses only signals available in the snapshot and carries one reason code and one human action.

In [2]:
eligible['ctr_gap_positive'] = eligible['ctr_gap'].clip(lower=0)
eligible['baseline_action_score'] = np.log1p(eligible['impressions_90d']) * eligible['ctr_gap_positive']
queue = eligible[eligible['baseline_action_score'] > 0].copy()
queue['reason_code'] = 'low_ctr_visible_page'
queue['action_label'] = 'review title, snippet, and search-intent fit'
queue = queue.sort_values(['baseline_action_score', 'impressions_90d'], ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', np.arange(1, len(queue) + 1))

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/content_refresh_anonymized.csv').exists()), Path.cwd())
output_dir = repo_root / 'work/outputs'
output_dir.mkdir(parents=True, exist_ok=True)
queue_columns = [
    'rank', 'content_id', 'client_id', 'position_band', 'impressions_90d',
    'avg_position', 'ctr', 'expected_ctr', 'ctr_gap_positive',
    'baseline_action_score', 'reason_code', 'action_label'
]
queue[queue_columns].to_csv(output_dir / 'baseline_action_score.csv', index=False)

def precision_at_k(frame, k):
    return float(frame.head(k)['is_declining_proxy'].mean())

metrics = {
    'rows_scored': int(len(eligible)),
    'rows_prioritized': int(len(queue)),
    'proxy_base_rate': float(eligible['is_declining_proxy'].mean()),
    'precision_at_20': precision_at_k(queue, 20),
    'precision_at_50': precision_at_k(queue, 50),
    'score_inputs': ['impressions_90d', 'avg_position', 'ctr'],
}
(output_dir / 'baseline_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))
print(f"Wrote {output_dir / 'baseline_action_score.csv'}")
queue[queue_columns].head(10)


{
  "rows_scored": 16337,
  "rows_prioritized": 7961,
  "proxy_base_rate": 0.602619820040399,
  "precision_at_20": 0.7,
  "precision_at_50": 0.68,
  "score_inputs": [
    "impressions_90d",
    "avg_position",
    "ctr"
  ]
}
Wrote C:\Users\Acer\source\repos\FlyrankAI\flyrankAI\work\outputs\baseline_action_score.csv


,rank,content_id,client_id,position_band,impressions_90d,avg_position,ctr,expected_ctr,ctr_gap_positive,baseline_action_score,reason_code,action_label
0,1,content_c8e9d6ab9013,client_19581e27de,page_1,208678,9.7,0.00,0.24,0.24,2.939653,low_ctr_visible_page,"review title, snippet, and search-intent fit"
1,2,content_453722754fea,client_f369cb89fc,page_1,140079,7.6,0.01,0.24,0.23,2.725493,low_ctr_visible_page,"review title, snippet, and search-intent fit"
2,3,content_39881853ef0c,client_f369cb89fc,page_1,112434,7.2,0.01,0.24,0.23,2.674930,low_ctr_visible_page,"review title, snippet, and search-intent fit"
3,4,content_c84a0ab98e90,client_f369cb89fc,page_1,223271,7.8,0.03,0.24,0.21,2.586391,low_ctr_visible_page,"review title, snippet, and search-intent fit"
4,5,content_0919dd345d80,client_4e07408562,page_1,119217,7.0,0.02,0.24,0.22,2.571516,low_ctr_visible_page,"review title, snippet, and search-intent fit"
5,6,content_d274ac4158ef,client_4e07408562,page_1,65138,6.8,0.01,0.24,0.23,2.549384,low_ctr_visible_page,"review title, snippet, and search-intent fit"
6,7,content_e5f459e737b7,client_f369cb89fc,page_1,56363,5.9,0.01,0.24,0.23,2.516105,low_ctr_visible_page,"review title, snippet, and search-intent fit"
7,8,content_c1fe78bc4e37,client_19581e27de,page_1,134055,7.5,0.03,0.24,0.21,2.479263,low_ctr_visible_page,"review title, snippet, and search-intent fit"
8,9,content_339b357d04c7,client_bbb965ab0c,page_1,46879,3.7,0.01,0.24,0.23,2.473730,low_ctr_visible_page,"review title, snippet, and search-intent fit"
9,10,content_65114d89496d,client_19581e27de,page_1,72631,6.5,0.02,0.24,0.22,2.462495,low_ctr_visible_page,"review title, snippet, and search-intent fit"


## 3. Top-20 review

The review below is a decision audit, not an automatic edit list. Each line includes the proposed action, why the page appears, confidence, and a specific condition that could invalidate the recommendation. Pseudonymous IDs are retained only so the run is reproducible.

In [3]:
top20 = queue.head(20).copy()
top20['confidence_note'] = np.where(
    (top20['impressions_90d'] >= 5000) & (top20['ctr_gap_positive'] >= 0.10),
    'high measured opportunity; still requires human review',
    'medium measured opportunity; inspect query mix first',
)
top20['what_would_make_it_wrong'] = np.select(
    [
        top20['position_band'].isin(['top_3', 'page_1']),
        top20['position_band'].eq('striking'),
    ],
    [
        'brand/navigation intent or SERP features make the band median incomparable',
        'query mix differs from peers; ranking improvement may matter more than the snippet',
    ],
    default='low rank explains low CTR; a snippet change alone may not help',
)
review_columns = [
    'rank', 'content_id', 'action_label', 'reason_code', 'confidence_note',
    'what_would_make_it_wrong', 'impressions_90d', 'avg_position', 'ctr', 'expected_ctr'
]
pd.set_option('display.max_colwidth', 90)
top20[review_columns]


,rank,content_id,action_label,reason_code,confidence_note,what_would_make_it_wrong,impressions_90d,avg_position,ctr,expected_ctr
0,1,content_c8e9d6ab9013,"review title, snippet, and search-intent fit",low_ctr_visible_page,high measured opportunity; still requires human review,brand/navigation intent or SERP features make the band median incomparable,208678,9.7,0.00,0.24
1,2,content_453722754fea,"review title, snippet, and search-intent fit",low_ctr_visible_page,high measured opportunity; still requires human review,brand/navigation intent or SERP features make the band median incomparable,140079,7.6,0.01,0.24
2,3,content_39881853ef0c,"review title, snippet, and search-intent fit",low_ctr_visible_page,high measured opportunity; still requires human review,brand/navigation intent or SERP features make the band median incomparable,112434,7.2,0.01,0.24
3,4,content_c84a0ab98e90,"review title, snippet, and search-intent fit",low_ctr_visible_page,high measured opportunity; still requires human review,brand/navigation intent or SERP features make the band median incomparable,223271,7.8,0.03,0.24
4,5,content_0919dd345d80,"review title, snippet, and search-intent fit",low_ctr_visible_page,high measured opportunity; still requires human review,brand/navigation intent or SERP features make the band median incomparable,119217,7.0,0.02,0.24
5,6,content_d274ac4158ef,"review title, snippet, and search-intent fit",low_ctr_visible_page,high measured opportunity; still requires human review,brand/navigation intent or SERP features make the band median incomparable,65138,6.8,0.01,0.24
6,7,content_e5f459e737b7,"review title, snippet, and search-intent fit",low_ctr_visible_page,high measured opportunity; still requires human review,brand/navigation intent or SERP features make the band median incomparable,56363,5.9,0.01,0.24
7,8,content_c1fe78bc4e37,"review title, snippet, and search-intent fit",low_ctr_visible_page,high measured opportunity; still requires human review,brand/navigation intent or SERP features make the band median incomparable,134055,7.5,0.03,0.24
8,9,content_339b357d04c7,"review title, snippet, and search-intent fit",low_ctr_visible_page,high measured opportunity; still requires human review,brand/navigation intent or SERP features make the band median incomparable,46879,3.7,0.01,0.24
9,10,content_65114d89496d,"review title, snippet, and search-intent fit",low_ctr_visible_page,high measured opportunity; still requires human review,brand/navigation intent or SERP features make the band median incomparable,72631,6.5,0.02,0.24


## 4. Weak picks + leakage check

A deliberately skeptical check flags top-20 rows that are **not declining under the teaching proxy**. They are weak picks for that proxy, although they may still be legitimate CTR opportunities. This distinction is important: the baseline ranks a measured CTR shortfall; it does not claim a refresh will cause recovery. The score uses only impressions, position, and CTR. Label-derived trend fields, recent/previous comparison windows, and pseudonymous IDs are excluded from scoring.

In [4]:
score_inputs = {'impressions_90d', 'avg_position', 'ctr'}
prohibited = {
    'trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d',
    'is_declining_proxy', 'content_id', 'client_id'
}
assert score_inputs.isdisjoint(prohibited)
weak_picks = top20.loc[
    top20['is_declining_proxy'].eq(0),
    ['rank', 'content_id', 'impressions_90d', 'avg_position', 'ctr', 'expected_ctr']
].copy()
weak_picks['why_weak_for_proxy'] = 'not declining in the observed comparison window'
print(f'Leakage check passed. Weak picks in top 20: {len(weak_picks)}')
weak_picks


Leakage check passed. Weak picks in top 20: 6


,rank,content_id,impressions_90d,avg_position,ctr,expected_ctr,why_weak_for_proxy
3,4,content_c84a0ab98e90,223271,7.8,0.03,0.24,not declining in the observed comparison window
5,6,content_d274ac4158ef,65138,6.8,0.01,0.24,not declining in the observed comparison window
8,9,content_339b357d04c7,46879,3.7,0.01,0.24,not declining in the observed comparison window
10,11,content_b115f7c74779,123469,8.0,0.03,0.24,not declining in the observed comparison window
15,16,content_36ff89c8214e,295097,7.3,0.05,0.24,not declining in the observed comparison window
17,18,content_f6ae0f36d70d,44860,9.0,0.02,0.24,not declining in the observed comparison window


## Self-check

- [x] Two signal checks show bucket tables, `n`, and explicit verdicts
- [x] One transparent rule produces a score, one reason code, and one action label
- [x] The notebook writes `work/outputs/baseline_action_score.csv` and a metrics receipt
- [x] All top 20 rows include an action, reason, confidence note, and failure condition
- [x] Label-derived and future-window inputs are excluded from the score
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries appear in the output